# <center>**05_Create_subsets**</center>

### Table of Contents

0. **Prepare Environment**

1. **Notebook Overview**
   - Goal of the notebook.
   - Subset design.
   - Inputs and expected outputs.

2. **Data Loading and Exploration**
   - Load metadata.
   - Inspect projection availability and station distribution.
   - Review class balance.

3. **Subset Definition**
   - Define train/test subsets by acquisition station.
   - Specify the number of clinical episodes per subset.

4. **Subset Creation**
   - Generate subset directories and metadata files.
   - Validate the number of saved projection files.

5. **Summary and Conclusions**
   - Final subset composition.
   - Readiness of the data for Experiment 2.

### **0. Prepare environment**

In [23]:
import os
import numpy as np
import pandas as pd
import shutil
from sklearn.preprocessing import OrdinalEncoder
import warnings
from pathlib import Path
from contextlib import redirect_stdout
from io import StringIO
import shutil

# Filter warnings
warnings.filterwarnings("ignore")

### **1. Notebook Overview**

### **2. Data Loading and Exploration**

In [15]:
# Load the main labels table
df_data = pd.read_csv("Dataset/Main_dataset/df_all_data_unique.csv")
print(df_data.shape)

# Preview column names without printing data rows
df_data.head(0)

# Group by clinical episode and concatenate available projection labels
df_grouped = (
    df_data.groupby("epi_cod")["proje"]
    .agg(", ".join)
    .reset_index()
)

# Keep only episodes containing both standard projections: RE and RI
df_two_projections = df_grouped[df_grouped["proje"] == "RE, RI"]

# Extract the list of episode IDs with both projections available
epi_cod_two_projections = df_two_projections["epi_cod"].tolist()
len(epi_cod_two_projections)

(4978, 19)


2474

In [16]:
# Summarize the number of samples by acquisition station and class label
grouped_counts = (
    df_2_proje.groupby(["station_name", "label_CalTend"])
    .size()
    .reset_index(name="count")
)

# Create a pivot table for easier inspection of class distribution by station
df_pivot = grouped_counts.pivot_table(
    index="station_name",
    columns="label_CalTend",
    values="count",
    fill_value=0
)

# Convert image counts to episode counts
# Each clinical episode contributes two images (RE and RI)
df_pivot / 2

label_CalTend,No,Sí
station_name,,
CZC1198HNF,0.0,2.0
CZC1198HNN,489.0,554.0
HRJ SALA 2,6.0,18.0
HRJCDXVILLA,33.0,11.0
HRJCDXVILLA24,26.0,8.0
MININT-VSAMOTD,197.0,170.0
PRIMO R,78.0,17.0
RADIOLOGIA-HP,405.0,460.0


### **3. Subset Definition**

In [17]:
# Definition of the episode-based subsets used in the study
# Format:
#   subset_name: [acquisition_station, number_of_clinical_episodes]
subsets_dict = {
    "TC_Canon_CZC_2proj_HURJC_train": ["CZC1198HNN", 870],
    "TC_Canon_RAD_2proj_HURJC_train": ["RADIOLOGIA-HP", 706],
    "TC_Canon_CZC_2proj_HURJC_test":  ["CZC1198HNN", 98],
    "TC_Canon_RAD_2proj_HURJC_test":  ["RADIOLOGIA-HP", 100],
}

### **4. Subset Creation**

In [18]:
def stratify_subset_exp3(df_data, epi_cod, label, station_name, n_subset, col_rx_id, image_directory, subsets_dir, subset_name, seed=42):
    # Copy dataframe
    df = df_data.copy()

    # Encode binary labels if needed
    if df[label].dtype != 'int':
        encoder = OrdinalEncoder()
        df[label] = encoder.fit_transform(df[[label]]).astype(int)

    # Filter by station name
    df = df[df['station_name'] == station_name]

    # Groupby epi_cod
    df_groupby_epi_cod = df.groupby(epi_cod).first()

    # Select index of given station_name and each class
    idx_station_l0 = df_groupby_epi_cod[df_groupby_epi_cod[label] == 0].index
    idx_station_l1 = df_groupby_epi_cod[df_groupby_epi_cod[label] == 1].index

    # Generate random index
    high_l0 = len(idx_station_l0)
    high_l1 = len(idx_station_l1)

    np.random.seed(seed)
    idx_list_l0 = np.random.choice(high_l0, size=int(n_subset / 2), replace=False)
    idx_list_l1 = np.random.choice(high_l1, size=int(n_subset / 2), replace=False)

    idx_l0 = np.array(idx_station_l0)[idx_list_l0]
    idx_l1 = np.array(idx_station_l1)[idx_list_l1]
    idx_selected = np.concatenate([idx_l0, idx_l1])
    np.random.shuffle(idx_selected)

    df_subset = df_groupby_epi_cod.loc[idx_selected].copy()
    df_subset_rx_level = df[df[epi_cod].isin(idx_selected)].copy()

    X_names = [X_name[:-2] for X_name in df_subset[col_rx_id].values]

    os.makedirs(subsets_dir, exist_ok=True)
    subset_dir = os.path.join(subsets_dir, subset_name)
    os.makedirs(subset_dir, exist_ok=False)

    subset_dir_all = subset_dir.replace('_2proj', '')
    subset_dir_RE = subset_dir.replace('2proj', '1proje_RE')
    subset_dir_RI = subset_dir.replace('2proj', '1proje_RI')

    os.makedirs(subset_dir_all, exist_ok=False)
    os.makedirs(subset_dir_RE, exist_ok=False)
    os.makedirs(subset_dir_RI, exist_ok=False)

    missing = 0
    copied = 0
    missing_epi_cod = []

    for X_name in X_names:
        X_path_RE = os.path.join(image_directory, X_name + 'RE_cropped.npy')
        X_path_RI = os.path.join(image_directory, X_name + 'RI_cropped.npy')

        X_RE_new_path = os.path.join(subset_dir_RE, X_name + 'RE_cropped.npy')
        X_RI_new_path = os.path.join(subset_dir_RI, X_name + 'RI_cropped.npy')

        X_all_RE_new_path = os.path.join(subset_dir_all, X_name + 'RE_cropped.npy')
        X_all_RI_new_path = os.path.join(subset_dir_all, X_name + 'RI_cropped.npy')

        X_2proje_new_path = os.path.join(subset_dir, X_name + '2proje_cropped.npy')

        if os.path.exists(X_path_RE) and os.path.exists(X_path_RI) and not os.path.exists(X_2proje_new_path):
            X_RE = np.load(X_path_RE)
            X_RI = np.load(X_path_RI)
            X_3 = np.zeros_like(X_RI)

            # Comprobación de formas
            if X_RE.shape == X_RI.shape == X_3.shape:
                try:
                    X_2proje = np.stack([X_RE[:, :, 1], X_RI[:, :, 1], X_3[:, :, 1]], axis=-1)
                    np.save(X_2proje_new_path, X_2proje)

                    shutil.copy(X_path_RE, X_RE_new_path)
                    shutil.copy(X_path_RE, X_all_RE_new_path)
                    shutil.copy(X_path_RI, X_RI_new_path)
                    shutil.copy(X_path_RI, X_all_RI_new_path)

                    copied += 1
                except Exception as e:
                    print(f"Error stacking for {X_name}: {e}")
                    missing += 1
                    missing_epi_cod.append(X_name[:-3])
            else:
                print(f"Shape mismatch for {X_name}: RE={X_RE.shape}, RI={X_RI.shape}, Z={X_3.shape}")
                missing += 1
                missing_epi_cod.append(X_name[:-3])
        else:
            if not os.path.exists(X_path_RE):
                print(f'File not exists: {X_path_RE}')
            elif not os.path.exists(X_path_RI):
                print(f'File not exists: {X_path_RI}')
            elif os.path.exists(X_2proje_new_path):
                print(f'File already exists: {X_2proje_new_path}')
            missing += 1
            missing_epi_cod.append(X_name[:-3])

    print(f'Copied:  {copied}  files')
    print(f'Missing: {missing} files')

    # Drop missing images
    df_subset_rx_level = df_subset_rx_level[~df_subset_rx_level[epi_cod].isin(missing_epi_cod)]

    # Save df_subset dataframe
    df_subset_name = os.path.join(subsets_dir, subset_name + '.csv')
    df_subset_rx_level.to_csv(df_subset_name, index=True)

    # Return the remaining dataframe
    df_data_groupby_epi_cod = df_data.groupby(epi_cod).first()
    idx_remaining = np.setdiff1d(np.array(df_data_groupby_epi_cod.index.values), idx_selected)
    df_remaining = df_data[df_data[epi_cod].isin(idx_remaining)].copy()
    df_remaining = df_remaining.reset_index(drop=True)

    print(f'Subset: {subset_name}')
    print(f'Saved:  {subset_dir}')
    print(f'Images: {len(X_names)/2}')
    print('Done!\n')

    return df_remaining

In [20]:
# Create the study subsets and validate the expected number of saved files.
# Each clinical episode is expected to include two projections: RE and RI.

df_ = df_data.copy()

images_root = Path("big_volume/Cropped_images_unet")
subsets_root = Path("big_volume/Subsets/LRM")
files_per_episode = 2

summary_rows = []

for idx, (subset_name, (station_name, n_episodes)) in enumerate(subsets_dict.items(), start=1):
    print(f"Generating subset... {idx}/{len(subsets_dict)}")

    subset_dir = subsets_root / subset_name

    # Remove any previous subset folder to avoid stale counts
    if subset_dir.exists():
        shutil.rmtree(subset_dir)

    # Suppress verbose internal messages from the helper function
    with redirect_stdout(StringIO()):
        df_ = stratify_subset_exp3(
            df_data=df_,
            epi_cod="epi_cod",
            label="label_CalTend",
            station_name=station_name,
            n_subset=n_episodes,
            col_rx_id="rx_cod",
            image_directory=str(images_root),
            subsets_dir=str(subsets_root),
            subset_name=subset_name,
            seed=42
        )

    # Count saved NumPy files
    saved_files = len(list(subset_dir.rglob("*.npy")))
    expected_files = n_episodes * files_per_episode
    missing_files = max(expected_files - saved_files, 0)
    status = "OK" if saved_files == expected_files else "CHECK"

    print(f"Copied: {saved_files} files")
    print(f"Missing: {missing_files} files")
    print(f"Subset: {subset_name}")
    print(f"Saved: {subset_dir}")
    print(f"Episodes: {n_episodes}")
    print(f"Status: {status}")
    print("Done!\n")

    summary_rows.append({
        "subset_name": subset_name,
        "station_name": station_name,
        "episodes": n_episodes,
        "expected_files": expected_files,
        "saved_files": saved_files,
        "missing_files": missing_files,
        "status": status,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

Generating subset... 1/4
Copied: 1740 files
Missing: 0 files
Subset: TC_Canon_CZC_2proj_HURJC_train
Saved: big_volume/Subsets/LRM/TC_Canon_CZC_2proj_HURJC_train
Episodes: 870
Status: OK
Done!

Generating subset... 2/4
Copied: 1412 files
Missing: 0 files
Subset: TC_Canon_RAD_2proj_HURJC_train
Saved: big_volume/Subsets/LRM/TC_Canon_RAD_2proj_HURJC_train
Episodes: 706
Status: OK
Done!

Generating subset... 3/4
Copied: 196 files
Missing: 0 files
Subset: TC_Canon_CZC_2proj_HURJC_test
Saved: big_volume/Subsets/LRM/TC_Canon_CZC_2proj_HURJC_test
Episodes: 98
Status: OK
Done!

Generating subset... 4/4
Copied: 200 files
Missing: 0 files
Subset: TC_Canon_RAD_2proj_HURJC_test
Saved: big_volume/Subsets/LRM/TC_Canon_RAD_2proj_HURJC_test
Episodes: 100
Status: OK
Done!



### **5. Summary and Conclusions**